# Session 05 · Database Commands

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Setup

In [2]:
from campus_data import seed
from commands import run
from mongo_conn import backend, get_db
print("Backend:", backend())
db = get_db("campus_cmds")
for c in db.list_collection_names():
    db[c].drop()
print(seed(db))

Backend: in-memory mongomock


{'students': 10, 'batches': 3, 'payments': 12}


## Example 1 - health check

In [3]:
print(run(db, "ping"), run(db, "buildInfo")["version"])

{'ok': 1.0} mongomock (emulated)


## Example 2 - database statistics

In [4]:
stats = run(db, "dbStats")
print({k: stats[k] for k in ("collections", "objects", "dataSize", "avgObjSize", "indexes")})

{'collections': 3, 'objects': 25, 'dataSize': 3655, 'avgObjSize': 146, 'indexes': 3}


## Example 3 - collection statistics

In [5]:
cs = run(db, "collStats", "students")
print(cs["count"], "docs,", cs["size"], "bytes, avg", cs["avgObjSize"], "bytes")
assert cs["count"] == 10

10 docs, 2151 bytes, avg 215 bytes


## Example 4 - metadata

In [6]:
print([c["name"] for c in run(db, "listCollections")["cursor"]["firstBatch"]])

['batches', 'payments', 'students']


## Example 5 - count and distinct as commands

In [7]:
print(run(db, "count", "students", query={"city": "Karachi"})["n"])
print(run(db, "distinct", "students", key="batch")["values"])

5
['AI-01', 'DS-01', 'DS-02']


## Example 6 - a simple capacity report

In [8]:
for name in sorted(db.list_collection_names()):
    c = run(db, "collStats", name)
    print(f"{name:<10} {c['count']:>4} docs  {c['size']:>6} bytes")

batches       3 docs     458 bytes
payments     12 docs    1046 bytes
students     10 docs    2151 bytes
